# PPO with Continuous Action Space for Trade Execution

## Overview
This notebook implements Proximal Policy Optimization (PPO) with continuous action space for trade execution.
PPO is a state-of-the-art on-policy algorithm that clips policy updates to keep them within a trust region.

## Key Features:
- **Algorithm**: PPO (Proximal Policy Optimization)
- **Action Space**: Continuous (Box)
- **Policy Types**:
  - Standard Normal Gaussian policy
  - Dirichlet policy for simplex-constrained actions

- **Environment Types**:
  - Noise-based market simulation
  - Can be adapted to flow and strategic environments

## References
- PPO Paper: Schulman et al., "Proximal Policy Optimization Algorithms" (2017)
- CleanRL implementation: https://docs.cleanrl.dev/rl-algorithms/ppo/

## Cell 1: Imports and Setup

In [ ]:
import os
import random
import time
from dataclasses import dataclass
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions.normal import Normal
from torch.distributions.dirichlet import Dirichlet
from torch.utils.tensorboard import SummaryWriter
import sys

# Setup paths for imports
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from simulation.market_gym import Market

print(f"Project root: {project_root}")
print(f"Imports initialized")

## Cell 2: Configuration and Arguments

In [ ]:
@dataclass
class Args:
    """PPO training configuration"""
    exp_name: str = "ppo_continuous_action"
    seed: int = 0
    torch_deterministic: bool = True
    cuda: bool = True
    track: bool = False
    wandb_project_name: str = "cleanRL"
    wandb_entity: str = None
    capture_video: bool = False
    save_model: bool = True
    upload_model: bool = False
    hf_entity: str = ""
    env_id: str = "Market"
    total_timesteps: int = 200 * 70 * 100
    learning_rate: float = 1e-2
    num_envs: int = 70
    num_steps: int = 100
    anneal_lr: bool = True
    gamma: float = 1.0
    gae_lambda: float = 1.0
    num_minibatches: int = 1
    update_epochs: int = 1
    norm_adv: bool = True
    clip_coef: float = 0.1
    clip_vloss: bool = False
    ent_coef: float = 0.0
    vf_coef: float = 0.5
    max_grad_norm: float = 100.0
    target_kl: float = None
    batch_size: int = 0
    minibatch_size: int = 0
    num_iterations: int = 0

MARKET_CONFIG = {
    'volume': 20,
    'market_env': 'noise',
    'seed': 0
}

args = Args(
    total_timesteps=200 * 70 * 100,
    num_envs=70,
    num_steps=100
)

args.batch_size = int(args.num_envs * args.num_steps)
args.minibatch_size = int(args.batch_size // args.num_minibatches)
args.num_iterations = args.total_timesteps // args.batch_size

print(f"Configuration: batch_size={args.batch_size}, iterations={args.num_iterations}")

## Cell 3: Device Setup

In [ ]:
random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)
torch.backends.cudnn.deterministic = args.torch_deterministic

device = torch.device("cuda" if torch.cuda.is_available() and args.cuda else "cpu")
print(f"Device: {device}")

run_name = f"{args.env_id}__{args.exp_name}__{args.seed}__{int(time.time())}_{MARKET_CONFIG['market_env']}_{MARKET_CONFIG['volume']}"
print(f"Run name: {run_name}")

## Cell 4: Neural Network Layers and Agent Classes

In [ ]:
def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    torch.nn.init.orthogonal_(layer.weight, std)
    torch.nn.init.constant_(layer.bias, bias_const)
    return layer

class Agent(nn.Module):
    def __init__(self, envs):
        n_hidden_units = 128 
        super().__init__()
        self.critic = nn.Sequential(
            layer_init(nn.Linear(np.array(envs.single_observation_space.shape).prod(), n_hidden_units)),
            nn.Tanh(),
            layer_init(nn.Linear(n_hidden_units, n_hidden_units)),
            nn.Tanh(),
            layer_init(nn.Linear(n_hidden_units, 1), std=1.0),
        )
        self.actor_mean = nn.Sequential(
            layer_init(nn.Linear(np.array(envs.single_observation_space.shape).prod(), n_hidden_units)),
            nn.Tanh(),
            layer_init(nn.Linear(n_hidden_units, n_hidden_units)),
            nn.Tanh(),
            layer_init(nn.Linear(n_hidden_units, np.prod(envs.single_action_space.shape)), std=0.01),
        )
        self.actor_logstd = nn.Parameter(torch.zeros(1, np.prod(envs.single_action_space.shape)), requires_grad=True)

    def get_value(self, x):
        return self.critic(x)

    def get_action_and_value(self, x, action=None):
        action_mean = self.actor_mean(x)
        action_logstd = self.actor_logstd.expand_as(action_mean)
        action_std = torch.exp(action_logstd)
        probs = Normal(action_mean, action_std)
        if action is None:
            action = probs.sample()
        return action, probs.log_prob(action).sum(1), probs.entropy().sum(1), self.critic(x)

print("Agent class defined")

## Cell 5: Environment and Agent Setup

In [ ]:
def make_env(config):
    def thunk():
        return Market(config)
    return thunk

print(f"Setting up {args.num_envs} parallel environments...")
configs = [{
    'market_env': MARKET_CONFIG['market_env'],
    'execution_agent': 'rl_agent',
    'volume': MARKET_CONFIG['volume'],
    'seed': args.seed + s
} for s in range(args.num_envs)]

env_fns = [make_env(config) for config in configs]
envs = gym.vector.AsyncVectorEnv(env_fns=env_fns)

assert isinstance(envs.single_action_space, gym.spaces.Box), "Only continuous action space supported"

print(f"Observation space: {envs.single_observation_space}")
print(f"Action space: {envs.single_action_space}")

observation, info = envs.reset(seed=args.seed)

## Cell 6: Initialize Agent and Optimizer

In [ ]:
agent = Agent(envs).to(device)
print(f"Agent initialized on device: {device}")

optimizer = optim.Adam(agent.parameters(), lr=args.learning_rate, eps=1e-5)
print(f"Optimizer: Adam with learning rate {args.learning_rate}")

## Cell 7: Setup TensorBoard and Storage

In [ ]:
writer = SummaryWriter(f"runs_t200_std2/{run_name}")

hyperparams_str = "\\n".join([f"|{key}|{value}|" for key, value in vars(args).items()])
writer.add_text(
    "hyperparameters",
    "|param|value|\\n|-|-|\\n" + hyperparams_str,
)

# Storage setup
obs = torch.zeros((args.num_steps, args.num_envs) + envs.single_observation_space.shape).to(device)
actions = torch.zeros((args.num_steps, args.num_envs) + envs.single_action_space.shape).to(device)
logprobs = torch.zeros((args.num_steps, args.num_envs)).to(device)
rewards = torch.zeros((args.num_steps, args.num_envs)).to(device)
dones = torch.zeros((args.num_steps, args.num_envs)).to(device)
values = torch.zeros((args.num_steps, args.num_envs)).to(device)

print(f"TensorBoard: runs_t200_std2/{run_name}")
print(f"Storage allocated on {device}")

## Cell 8: Initialize Training State

In [ ]:
global_step = 0
start_time = time.time()
next_obs, _ = envs.reset(seed=args.seed)
next_obs = torch.Tensor(next_obs).to(device)
next_done = torch.zeros(args.num_envs).to(device)

print(f"Training initialized for {args.num_iterations} iterations")

## Cell 9: Main Training Loop

In [ ]:
for iteration in range(1, args.num_iterations + 1):
    print(f"Iteration {iteration}/{args.num_iterations}")
    
    if args.anneal_lr:
        frac = 1.0 - (iteration - 1.0) / args.num_iterations
        lrnow = frac * args.learning_rate
        optimizer.param_groups[0]["lr"] = lrnow
    
    returns = []
    times = []
    drifts = []
    
    for step in range(0, args.num_steps):
        global_step += args.num_envs
        obs[step] = next_obs
        dones[step] = next_done

        with torch.no_grad():
            action, logprob, _, value = agent.get_action_and_value(next_obs)
            values[step] = value.flatten()
        actions[step] = action
        logprobs[step] = logprob

        next_obs, reward, terminations, truncations, infos = envs.step(action.cpu().numpy())
        next_done = np.logical_or(terminations, truncations)
        rewards[step] = torch.tensor(reward).to(device).view(-1)
        next_obs, next_done = torch.Tensor(next_obs).to(device), torch.Tensor(next_done).to(device)

        if "final_info" in infos:
            for info in infos["final_info"]:
                if info is not None:
                    returns.append(info.get('cum_reward', info.get('reward', 0)))
                    times.append(info.get('time', 0))
                    drifts.append(info.get('drift', 0))
    
    if returns:
        writer.add_scalar("charts/return", np.mean(returns), global_step)
        writer.add_scalar("charts/time", np.mean(times), global_step)
        writer.add_scalar("charts/drift", np.mean(drifts), global_step)
        print(f"  Episode Return: {np.mean(returns):.4f}")

## Cell 10: Advantage and Policy Updates

In [ ]:
    with torch.no_grad():
        next_value = agent.get_value(next_obs).reshape(1, -1)
        advantages = torch.zeros_like(rewards).to(device)
        lastgaelam = 0
        for t in reversed(range(args.num_steps)):
            if t == args.num_steps - 1:
                nextnonterminal = 1.0 - next_done
                nextvalues = next_value
            else:
                nextnonterminal = 1.0 - dones[t + 1]
                nextvalues = values[t + 1]
            delta = rewards[t] + args.gamma * nextvalues * nextnonterminal - values[t]
            advantages[t] = lastgaelam = delta + args.gamma * args.gae_lambda * nextnonterminal * lastgaelam
        returns_gae = advantages + values

    b_obs = obs.reshape((-1,) + envs.single_observation_space.shape)
    b_logprobs = logprobs.reshape(-1)
    b_actions = actions.reshape((-1,) + envs.single_action_space.shape)
    b_advantages = advantages.reshape(-1)
    b_returns = returns_gae.reshape(-1)
    b_values = values.reshape(-1)

    b_inds = np.arange(args.batch_size)
    clipfracs = []
    
    for epoch in range(args.update_epochs):
        np.random.shuffle(b_inds)
        for start in range(0, args.batch_size, args.minibatch_size):
            end = start + args.minibatch_size
            mb_inds = b_inds[start:end]

            _, newlogprob, entropy, newvalue = agent.get_action_and_value(b_obs[mb_inds], b_actions[mb_inds])
            logratio = newlogprob - b_logprobs[mb_inds]
            ratio = logratio.exp()

            with torch.no_grad():
                old_approx_kl = (-logratio).mean()
                approx_kl = ((ratio - 1) - logratio).mean()
                clipfracs += [((ratio - 1.0).abs() > args.clip_coef).float().mean().item()]

            mb_advantages = b_advantages[mb_inds]
            if args.norm_adv:
                mb_advantages = (mb_advantages - mb_advantages.mean()) / (mb_advantages.std() + 1e-8)

            pg_loss1 = -mb_advantages * ratio
            pg_loss2 = -mb_advantages * torch.clamp(ratio, 1 - args.clip_coef, 1 + args.clip_coef)
            pg_loss = torch.max(pg_loss1, pg_loss2).mean()

            newvalue = newvalue.view(-1)
            if args.clip_vloss:
                v_loss_unclipped = (newvalue - b_returns[mb_inds]) ** 2
                v_clipped = b_values[mb_inds] + torch.clamp(
                    newvalue - b_values[mb_inds],
                    -args.clip_coef,
                    args.clip_coef,
                )
                v_loss_clipped = (v_clipped - b_returns[mb_inds]) ** 2
                v_loss_max = torch.max(v_loss_unclipped, v_loss_clipped)
                v_loss = 0.5 * v_loss_max.mean()
            else:
                v_loss = 0.5 * ((newvalue - b_returns[mb_inds]) ** 2).mean()

            entropy_loss = entropy.mean()
            loss = pg_loss - args.ent_coef * entropy_loss + v_loss * args.vf_coef

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(agent.parameters(), args.max_grad_norm)
            optimizer.step()

        if args.target_kl is not None and approx_kl > args.target_kl:
            break
    
    y_pred, y_true = b_values.cpu().numpy(), b_returns.cpu().numpy()
    var_y = np.var(y_true)
    explained_var = np.nan if var_y == 0 else 1 - np.var(y_true - y_pred) / var_y

    writer.add_scalar("charts/learning_rate", optimizer.param_groups[0]["lr"], global_step)
    writer.add_scalar("losses/value_loss", v_loss.item(), global_step)
    writer.add_scalar("losses/policy_loss", pg_loss.item(), global_step)
    writer.add_scalar("losses/entropy", entropy_loss.item(), global_step)
    writer.add_scalar("losses/old_approx_kl", old_approx_kl.item(), global_step)
    writer.add_scalar("losses/approx_kl", approx_kl.item(), global_step)
    writer.add_scalar("losses/clipfrac", np.mean(clipfracs), global_step)
    writer.add_scalar("losses/explained_variance", explained_var, global_step)
    
    sps = int(global_step / (time.time() - start_time))
    writer.add_scalar("charts/SPS", sps, global_step)
    print(f"  Steps/sec: {sps}")

## Cell 11: Save and Finalize

In [ ]:
if args.save_model:
    model_path = f"runs_t200_std2/{run_name}/{args.exp_name}.cleanrl_model"
    os.makedirs(os.path.dirname(model_path), exist_ok=True)
    torch.save(agent.state_dict(), model_path)
    print(f"Model saved to {model_path}")

envs.close()
writer.close()

total_time = time.time() - start_time
print(f"\\nTraining completed in {total_time:.2f}s")
print(f"Steps/second: {int(global_step / total_time)}")